<a href="https://colab.research.google.com/github/BeastHunter0041/csci_4170_s26/blob/main/01_nlp_pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Part A

In [ ]:
pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00


In [ ]:
from transformers import pipeline
from datasets import load_dataset
import evaluate
import time
from PIL import Image


In [ ]:

# Sentiment analysis pipeline
sentiment_pipe = pipeline("text-classification")

# Question Answering pipeline
question_pipe = pipeline("question-answering")

print(sentiment_pipe("I liked this movie."))

print(question_pipe(
    question="Who liked the movie?",
    context="John liked the movie."
))

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9998389482498169}]
{'score': 0.9949975609779358, 'start': 0, 'end': 4, 'answer': 'John'}


#Part B

In [ ]:
# QA pipeline
question_pipe = pipeline("question-answering")

# Load a small subset of the squad validation set
dataset = load_dataset("squad", split="validation[:100]")

# Load SQuAD evaluation metric
squad_metric = evaluate.load("squad")

predictions = []
references = []

start_time = time.time()

for ex in dataset:
    pred = question_pipe(
        question=ex["question"],
        context=ex["context"]
    )

    predictions.append({
        "id": ex["id"],
        "prediction_text": pred["answer"]
    })

    references.append({
        "id": ex["id"],
        "answers": ex["answers"]
    })

end_time = time.time()

results = squad_metric.compute(predictions=predictions, references=references)
avg_time = (end_time - start_time) / len(dataset)

print("QA Results on squad subset")
print(f"Exact Match: {results['exact_match']:.2f}")
print(f"F1 Score: {results['f1']:.2f}")
print(f"Average inference time per example: {avg_time:.4f} seconds")

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

QA Results on squad subset
Exact Match: 84.00
F1 Score: 89.00
Average inference time per example: 0.0080 seconds


#Part C

In [ ]:
# Compare two QA models on the same squad subset

qa_models = [
    "distilbert/distilbert-base-cased-distilled-squad",
    "deepset/roberta-base-squad2"
]

dataset = load_dataset("squad", split="validation[:100]")
squad_metric = evaluate.load("squad")

comparison_results = []

for model_id in qa_models:
    qa_pipe = pipeline("question-answering", model=model_id)

    predictions = []
    references = []

    start_time = time.time()

    for ex in dataset:
        pred = qa_pipe(
            question=ex["question"],
            context=ex["context"]
        )

        predictions.append({
            "id": ex["id"],
            "prediction_text": pred["answer"]
        })

        references.append({
            "id": ex["id"],
            "answers": ex["answers"]
        })

    end_time = time.time()

    metrics = squad_metric.compute(predictions=predictions, references=references)
    avg_latency = (end_time - start_time) / len(dataset)

    comparison_results.append({
        "model": model_id,
        "exact_match": round(metrics["exact_match"], 2),
        "f1": round(metrics["f1"], 2),
        "avg_latency_sec": round(avg_latency, 4)
    })

for row in comparison_results:
    print(row)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

{'model': 'distilbert/distilbert-base-cased-distilled-squad', 'exact_match': 84.0, 'f1': 89.0, 'avg_latency_sec': 0.0082}
{'model': 'deepset/roberta-base-squad2', 'exact_match': 97.0, 'f1': 98.23, 'avg_latency_sec': 0.0143}


In [ ]:
import pandas as pd

comparison_df = pd.DataFrame(comparison_results)
comparison_df

,model,exact_match,f1,avg_latency_sec
0,distilbert/distilbert-base-cased-distilled-squad,84.0,89.00,0.0082
1,deepset/roberta-base-squad2,97.0,98.23,0.0143


In [ ]:
# Concrete failure case
failure_example = {
    "question": "Who is the CEO?",
    "context": "Alice is sitting on the bench. Bob is sitting next to her."
}

for model_id in qa_models:
    qa_pipe = pipeline("question-answering", model=model_id)
    pred = qa_pipe(question=failure_example["question"], context=failure_example["context"])
    print(f"Model: {model_id}")
    print(f"Prediction: {pred['answer']}")
    print(f"Score: {pred['score']:.4f}")
    print()

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Model: distilbert/distilbert-base-cased-distilled-squad
Prediction: Bob
Score: 0.7527



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model: deepset/roberta-base-squad2
Prediction: Bob
Score: 0.0034



## Discussion

For the question-answering task, I compared `distilbert/distilbert-base-cased-distilled-squad` and `deepset/roberta-base-squad2` on a 100-example subset of the SQuAD validation set. The DistilBERT model was faster, with an average inference time of 0.0082 seconds per example, but it achieved lower performance with 84.0 Exact Match and 89.0 F1. In contrast, the RoBERTa model was slower at 0.0143 seconds per example, but it performed substantially better, reaching 97.0 Exact Match and 98.23 F1. This shows a clear tradeoff between efficiency and accuracy: the smaller model was faster, while the larger model produced more accurate answers.

A notable failure case appeared when both models were asked “Who is the CEO?” using a context that only stated that Alice and Bob were sitting on a bench. Since the context did not contain any information about a CEO, the models should ideally have recognized that the question was unsupported. Instead, both models predicted “Bob.” However, the confidence scores were very different: DistilBERT gave a relatively high score of 0.7527, while RoBERTa gave a very low score of 0.0034. This suggests that although both models can still return unsupported answers, the RoBERTa model was much less confident in its incorrect prediction.